In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from datetime import datetime

# Simulating a messy raw dataset
raw_data = [
    Row(order_id="ORD1001", cust_id="C901", order_datetime="2026-07-10 14:30:00", item="electronics", price=1200.0, country="Saudi Arabia"),
    Row(order_id="ORD1002", cust_id="C902", order_datetime="10-07-2026 15:00", item="Apparel", price=350.0, country="KSA"),
    Row(order_id="ORD1003", cust_id="C903", order_datetime="2026/07/11 09:15:00", item="HOME", price=-50.0, country="Saudi Arabia"),  # Negative price!
    Row(order_id="ORD1001", cust_id="C901", order_datetime="2026-07-10 14:35:00", item="electronics", price=1200.0, country="Saudi Arabia"), # Duplicate!
    Row(order_id="ORD1004", cust_id="C904", order_datetime="2026-07-12 11:00:00", item=None, price=150.0, country="KSA"), # Missing item!
    Row(order_id="ORD1005", cust_id="C905", order_datetime="2026-07-13 08:00:00", item="Beauty", price=None, country=None), # Missing price and country!
    Row(order_id="ORD1006", cust_id="C906", order_datetime="2028-01-01 12:00:00", item="Electronics", price=2500.0, country="KSA") # Future date anomaly!
]

df_bronze = spark.createDataFrame(raw_data)
display(df_bronze)

In [0]:
from pyspark.sql.functions import to_timestamp, col
df_clean1 = df_bronze.withColumn('order_datetime', (col('order_datetime').cast('timestamp')))

In [0]:
from pyspark.sql.functions import col, lit, coalesce, try_to_timestamp, date_format

# 1. Catch all three formats using coalesce
parsed_timestamp = coalesce(
    try_to_timestamp(col("order_datetime"), lit("yyyy-MM-dd HH:mm:ss")),
    try_to_timestamp(col("order_datetime"), lit("dd-MM-yyyy HH:mm")),
    try_to_timestamp(col("order_datetime"), lit("yyyy/MM/dd HH:mm:ss"))
)

# 2. Overwrite the column by formatting the parsed timestamp to drop the seconds
df_clean1 = df_bronze.withColumn(
    "order_datetime",
    date_format(parsed_timestamp, "yyyy-MM-dd HH:mm")
)

In [0]:
display(df_clean1)

In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

# Define window partitioned by order_id and ordered by order_datetime descending
window_spec = Window.partitionBy("order_id").orderBy(col("order_datetime").desc())

# Add row_number and filter to keep only the latest order_datetime per order_id
df_unique_orders = df_clean1.withColumn("rn", row_number().over(window_spec)).filter(col("rn") == 1).drop("rn")

display(df_unique_orders)

In [0]:
from pyspark.sql.functions import lower, coalesce, lit

df_clean_items = df_unique_orders.withColumn(
    "item",
    coalesce(lower(col("item")), lit("unknown"))
)

display(df_clean_items)

In [0]:
from pyspark.sql.functions import when

df_clean_country = df_clean_items.withColumn(
    "country",
    when(col("country").isin("Saudi Arabia", "KSA"), "KSA").otherwise("KSA")
)

display(df_clean_country)

In [0]:
df_valid_price = df_clean_country.filter((col("price").isNotNull()) & (col("price") >= 0))
display(df_valid_price)

In [0]:
from pyspark.sql.functions import to_timestamp, current_timestamp

df_valid_price_no_future = df_valid_price.filter(
    to_timestamp(col("order_datetime"), "yyyy-MM-dd HH:mm") <= current_timestamp()
)

display(df_valid_price_no_future)